# 16 - Dashboard Data Export
## ShopEase Europe | Sentiment Analysis Project - Phase 2 
**Objective:** Prepare a clean, business friendly dataset for the 
Power BI interactive dashboard, including a derived flag identifying 
reviews related to the delivery and driver complaint theme identified 
in notebook 13.

## Import Libaries

In [1]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Load the Dataset

In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'production_preprocessed_reviews.csv')

df = pd.read_csv(PROCESSED_DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Dataset loaded: 21,055 rows x 11 columns
Columns: ['review_id', 'cleaned_review', 'preprocessed_text', 'sentiment', 'country', 'product_category', 'year', 'month', 'rating', 'review_length', 'word_count']


## Add Delivery Complaint Flag
Flagging reviews that mention delivery or driver related terms, 
identified in notebook 13 as the primary driver of the 2019 to 2022 
sentiment decline, enabling this theme to be filtered directly in 
the dashboard.

In [3]:
delivery_terms = ['delivery', 'driver', 'delivered', 'drivers', 'courier', 'package', 'box', 'shipping']

def flag_delivery_complaint(text):
    text = str(text).lower()
    return any(term in text for term in delivery_terms)

df['delivery_related'] = df['cleaned_review'].apply(flag_delivery_complaint)

print(f"Reviews flagged as delivery related: {df['delivery_related'].sum():,} ({df['delivery_related'].mean()*100:.1f}%)")
print(f"\nDelivery related flag by sentiment:")
print(df.groupby('sentiment')['delivery_related'].mean().round(3) * 100)

Reviews flagged as delivery related: 8,933 (42.4%)

Delivery related flag by sentiment:
sentiment
negative    46.7
neutral     46.0
positive    31.3
Name: delivery_related, dtype: float64


## Build Dashboard Ready Dataset
Selecting and renaming columns for clarity, dropping fields not 
needed for business reporting.

In [4]:
dashboard_df = df[[
    'review_id', 'cleaned_review', 'sentiment', 'rating',
    'country', 'product_category', 'year', 'month',
    'review_length', 'delivery_related'
]].copy()

dashboard_df = dashboard_df.rename(columns={
    'cleaned_review': 'review_text',
    'product_category': 'category'
})

# Capitalise sentiment for cleaner display in Power BI
dashboard_df['sentiment'] = dashboard_df['sentiment'].str.capitalize()

print(f"Dashboard dataset shape: {dashboard_df.shape[0]:,} rows x {dashboard_df.shape[1]} columns")
print(f"\nColumns: {dashboard_df.columns.tolist()}")
display(dashboard_df.head())

Dashboard dataset shape: 21,055 rows x 10 columns

Columns: ['review_id', 'review_text', 'sentiment', 'rating', 'country', 'category', 'year', 'month', 'review_length', 'delivery_related']


,review_id,review_text,sentiment,rating,country,category,year,month,review_length,delivery_related
0,REV-50BCBCD9,"i registered on the website, tried to order a ...",Negative,1,US,Sports,2024,9,587,False
1,REV-6D2B2651,had multiple orders one turned up and driver h...,Negative,1,GB,Toys,2024,9,293,True
2,REV-F7E80372,i informed these reprobates that i would not b...,Negative,1,GB,Toys,2024,9,610,True
3,REV-ED2B173F,i have bought from amazon before and no proble...,Negative,1,AU,Sports,2024,9,449,False
4,REV-E48A7AB9,if i could give a lower rate i would! i cancel...,Negative,1,GB,Fashion,2024,9,535,False


## Export Dashboard Dataset

In [5]:
DASHBOARD_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'dashboard_data.csv')

dashboard_df.to_csv(DASHBOARD_PATH, index=False)

print(f"Dashboard dataset exported successfully to:")
print(DASHBOARD_PATH)
print(f"\nFile size: {os.path.getsize(DASHBOARD_PATH) / (1024*1024):.2f} MB")

Dashboard dataset exported successfully to:
c:\Users\User\OneDrive\Desktop\shopease-sentiment-analysis\data\processed\dashboard_data.csv

File size: 10.32 MB


## Summary

A dashboard ready dataset was exported with 21,055 rows and 10 columns, 
selected and renamed for clarity, review_id, review_text, sentiment, 
rating, country, category, year, month, review_length and a derived 
delivery_related flag.

This flag confirms the notebook 13 finding at scale, 42.4% of all 
reviews mention delivery related terms, with negative and neutral 
reviews mentioning it considerably more, 46.7% and 46.0% respectively, 
than positive reviews at 31.3%. This gives the Power BI dashboard a 
direct, filterable view into ShopEase Europe's most significant 
identified complaint theme.

Output saved to data/processed/dashboard_data.csv for import into 
Power BI.